# 66 - Scalar vs Anisotropic Adaptive R with `strict_conf_r4`

This notebook compares the previous scalar adaptive-R rule against the new anisotropic/diagonal adaptive-R rule using the same confidence evidence.

Candidate under test: `strict_conf_r4` from Notebook 64.

Configuration:

- `block_size = 21`
- `stride = 24`
- `search_radius = 4`
- `zncc_low = 0.45`
- `zncc_high = 0.95`
- `forward_backward_scale_px = 1.0`
- `motion_spread_scale_px = 2.0`
- `r_min_scale = 0.5`
- `r_max_scale = 20.0`
- `r_gamma = 1.5`

Comparisons:

- fixed-R strict baseline
- scalar adaptive-R: one confidence and one `r_scale`
- anisotropic adaptive-R: `confidence_theta -> r_scale_theta`, `confidence_length -> r_scale_length`

The MATLAB-compatible Kalman state order is `[x_sup, alpha]`, so internal `measurement_R_diag` is `[R_L/x, R_theta/alpha]`.

## 1. Imports and Setup

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import sys
from dataclasses import asdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.speckle_confidence import (
    SpeckleConfidenceConfig,
    anisotropic_confidence_to_r_scales,
    combine_anisotropic_confidence_metrics,
    confidence_to_r_scale,
)
from ultrasound_tracker.ultratimtrack_matlab_2state import (
    MatlabTwoStateKalmanConfig,
    run_matlab_2state_kalman,
)

VIDEO = ROOT / 'data' / 'raw' / 'UltraTimTrack_test.mp4'
ROI_PATH = ROOT / 'data' / 'rois' / 'UltraTimTrack_test_rois.json'
STRICT_BASELINE = ROOT / 'results' / 'strict_ultratimtrack_runs' / 'UltraTimTrack_test' / 'UltraTimTrack_test_strict_results.npz'
UTT_EXPORT = Path('/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat')
OUT = ROOT / 'results' / 'notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4'
CACHE = OUT / 'confidence_cache'
OVERLAY_DIR = OUT / 'low_confidence_overlays'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = SpeckleConfidenceConfig(
    block_size=21,
    stride=24,
    search_radius=4,
    zncc_low=0.45,
    zncc_high=0.95,
    confidence_floor=0.05,
    r_min_scale=0.5,
    r_max_scale=20.0,
    r_gamma=1.5,
    motion_spread_scale_px=2.0,
    forward_backward_scale_px=1.0,
)
FORCE_RECOMPUTE_CONFIDENCE = False
VALID_FRACTION_FLOOR = 0.25
FIRST_WINDOW_SECONDS = 30.0

for path in [VIDEO, ROI_PATH, STRICT_BASELINE, UTT_EXPORT]:
    print(path, 'OK' if path.exists() else 'MISSING')
print('output:', OUT)

/Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4 OK
/Users/grosbedou/PycharmProjects/NDORMS/data/rois/UltraTimTrack_test_rois.json OK
/Users/grosbedou/PycharmProjects/NDORMS/results/strict_ultratimtrack_runs/UltraTimTrack_test/UltraTimTrack_test_strict_results.npz OK
/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat OK
output: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4


## 2. Load Fixed Baseline and Video Metadata

In [2]:
def video_info(path: Path) -> tuple[float, int, int, int]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(path)
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ok, frame0 = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Could not read first frame from {path}')
    return fps, n_frames, int(frame0.shape[0]), int(frame0.shape[1])


def read_gray_frame(path: Path, frame_idx: int) -> np.ndarray:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Could not read frame {frame_idx} from {path}')
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame


with ROI_PATH.open('r', encoding='utf-8') as f:
    rois = json.load(f)
fascicle_roi = tuple(int(v) for v in rois['fascicle'])

fps, video_n_frames, image_h, image_w = video_info(VIDEO)
baseline = np.load(STRICT_BASELINE, allow_pickle=True)
n_all = min(len(baseline['frame']), video_n_frames)
n_first = min(n_all, int(round(float(FIRST_WINDOW_SECONDS) * fps)))
time_all = np.asarray(baseline['time_s'][:n_all], dtype=float)

super_lines = np.asarray(baseline['sup_apo_lines'][:n_all], dtype=float)
deep_lines = np.asarray(baseline['deep_apo_lines'][:n_all], dtype=float)
fascicle_prior = np.asarray(baseline['klt_prior_segments'][:n_all], dtype=float)
timtrack_alpha = np.asarray(baseline['timtrack_alpha_deg'][:n_all], dtype=float)

print({
    'fps': fps,
    'video_frames': video_n_frames,
    'image_shape': (image_h, image_w),
    'n_all': n_all,
    'n_first': n_first,
    'first_seconds': float(time_all[n_first - 1]),
    'fascicle_roi': fascicle_roi,
})

{'fps': 33.341, 'video_frames': 2667, 'image_shape': (562, 706), 'n_all': 2667, 'n_first': 1000, 'first_seconds': 29.963108485048437, 'fascicle_roi': (23, 78, 679, 218)}


## 3. Strict Confidence Helpers

This uses the local block-matching confidence rule from Notebook 64, plus simple geometry/aponeurosis/intersection scores for the anisotropic length channel.

In [3]:
def config_payload(config: SpeckleConfidenceConfig):
    payload = asdict(config)
    return {
        'block_size': payload['block_size'],
        'stride': payload['stride'],
        'search_radius': payload['search_radius'],
        'min_texture_variance': payload['min_texture_variance'],
        'zncc_low': payload['zncc_low'],
        'zncc_high': payload['zncc_high'],
        'confidence_floor': payload['confidence_floor'],
        'confidence_ceiling': payload['confidence_ceiling'],
        'r_min_scale': payload['r_min_scale'],
        'r_max_scale': payload['r_max_scale'],
        'r_gamma': payload['r_gamma'],
        'motion_spread_scale_px': payload['motion_spread_scale_px'],
        'forward_backward_scale_px': payload['forward_backward_scale_px'],
    }


def confidence_cache_path(n: int, config: SpeckleConfidenceConfig):
    encoded = json.dumps(config_payload(config), sort_keys=True).encode('utf-8')
    digest = hashlib.md5(encoded).hexdigest()[:10]
    return CACHE / f'strict_conf_r4_{digest}_first_{n}_frames.npz'


def line_y_at_x(line: np.ndarray, x: np.ndarray) -> np.ndarray:
    line = np.asarray(line, dtype=float).reshape(4)
    x1, y1, x2, y2 = line
    if abs(x2 - x1) < 1e-9:
        return np.full_like(x, np.nan, dtype=float)
    return y1 + (y2 - y1) * (x - x1) / (x2 - x1)


def line_angle_deg(line: np.ndarray) -> float:
    x1, y1, x2, y2 = np.asarray(line, dtype=float).reshape(4)
    return float(np.rad2deg(np.arctan2(y2 - y1, x2 - x1)))


def line_intersection(line_a: np.ndarray, line_b: np.ndarray):
    x1, y1, x2, y2 = np.asarray(line_a, dtype=float).reshape(4)
    x3, y3, x4, y4 = np.asarray(line_b, dtype=float).reshape(4)
    den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(den) < 1e-9:
        return None
    px = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / den
    py = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / den
    return float(px), float(py)


def normalize_angle_delta_deg(a: float, b: float) -> float:
    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan
    return float(abs(((a - b + 90.0) % 180.0) - 90.0))


def sample_roi_grid(roi_box, config: SpeckleConfidenceConfig, extra_border=4):
    x, y, w, h = [int(v) for v in roi_box]
    border = int(config.block_size // 2 + config.search_radius + extra_border)
    xs = np.arange(x + border, x + w - border + 1, int(config.stride), dtype=float)
    ys = np.arange(y + border, y + h - border + 1, int(config.stride), dtype=float)
    if len(xs) == 0 or len(ys) == 0:
        return np.empty((0, 2), dtype=np.float32)
    xx, yy = np.meshgrid(xs, ys)
    return np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)


def muscle_band_mask(points, super_line, deep_line, margin=3.0):
    points = np.asarray(points, dtype=float).reshape(-1, 2)
    y_super = line_y_at_x(super_line, points[:, 0])
    y_deep = line_y_at_x(deep_line, points[:, 0])
    upper = np.minimum(y_super, y_deep) + margin
    lower = np.maximum(y_super, y_deep) - margin
    return np.isfinite(upper) & np.isfinite(lower) & (points[:, 1] >= upper) & (points[:, 1] <= lower)


def robust_mad(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    med = np.median(values)
    return float(1.4826 * np.median(np.abs(values - med)))


def clip_confidence(values, config: SpeckleConfidenceConfig = CONFIG):
    return np.clip(values, float(config.confidence_floor), float(config.confidence_ceiling))

In [4]:
def match_patch_at(source, target, point, config: SpeckleConfidenceConfig):
    source = np.asarray(source, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    h, w = source.shape[:2]
    pr = int(config.block_size) // 2
    sr = int(config.search_radius)
    x, y = np.rint(np.asarray(point, dtype=np.float32)).astype(int)

    if x - pr < 0 or x + pr + 1 > w or y - pr < 0 or y + pr + 1 > h:
        return None
    if x - pr - sr < 0 or x + pr + sr + 1 > target.shape[1] or y - pr - sr < 0 or y + pr + sr + 1 > target.shape[0]:
        return None

    patch = source[y - pr : y + pr + 1, x - pr : x + pr + 1]
    if float(np.var(patch)) < float(config.min_texture_variance):
        return None

    sx0 = x - pr - sr
    sy0 = y - pr - sr
    search = target[sy0 : y + pr + sr + 1, sx0 : x + pr + sr + 1]
    response = cv2.matchTemplate(search, patch, cv2.TM_CCOEFF_NORMED)
    _, score, _, best_loc = cv2.minMaxLoc(response)
    matched = np.asarray([sx0 + best_loc[0] + pr, sy0 + best_loc[1] + pr], dtype=np.float32)
    return matched, float(score)


def block_match_forward_backward(prev_frame, curr_frame, points, config: SpeckleConfidenceConfig):
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    n = len(points)
    displacements = np.full((n, 2), np.nan, dtype=np.float32)
    zncc_scores = np.full(n, np.nan, dtype=np.float32)
    fb_errors = np.full(n, np.nan, dtype=np.float32)
    valid_mask = np.zeros(n, dtype=bool)

    for idx, point in enumerate(points):
        forward = match_patch_at(prev_frame, curr_frame, point, config)
        if forward is None:
            continue
        curr_point, score = forward
        if not np.isfinite(score):
            continue
        reverse = match_patch_at(curr_frame, prev_frame, curr_point, config)
        if reverse is not None:
            back_point, _ = reverse
            fb_errors[idx] = float(np.linalg.norm(back_point - point))
        displacements[idx] = curr_point - point
        zncc_scores[idx] = score
        valid_mask[idx] = True

    return displacements, zncc_scores, fb_errors, valid_mask


def local_confidence_from_matches(displacements, zncc_scores, fb_errors, valid_mask, config: SpeckleConfidenceConfig):
    displacements = np.asarray(displacements, dtype=float)
    zncc_scores = np.asarray(zncc_scores, dtype=float)
    fb_errors = np.asarray(fb_errors, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool) & np.isfinite(zncc_scores) & np.all(np.isfinite(displacements), axis=1)

    speckle_conf = np.zeros(len(zncc_scores), dtype=float)
    motion_conf = np.zeros(len(zncc_scores), dtype=float)
    fb_conf = np.zeros(len(zncc_scores), dtype=float)
    local_conf = np.zeros(len(zncc_scores), dtype=float)

    if not np.any(valid_mask):
        return local_conf, speckle_conf, motion_conf, fb_conf, np.full(2, np.nan), np.nan, np.nan

    speckle_raw = (zncc_scores[valid_mask] - float(config.zncc_low)) / max(float(config.zncc_high - config.zncc_low), 1e-12)
    speckle_conf[valid_mask] = clip_confidence(np.clip(speckle_raw, 0.0, 1.0), config)

    d_med = np.nanmedian(displacements[valid_mask], axis=0)
    residuals = np.linalg.norm(displacements - d_med, axis=1)
    motion_conf[valid_mask] = clip_confidence(
        np.exp(-residuals[valid_mask] / max(float(config.motion_spread_scale_px), 1e-12)),
        config,
    )

    fb_conf[valid_mask] = float(config.confidence_floor)
    finite_fb = valid_mask & np.isfinite(fb_errors)
    fb_conf[finite_fb] = clip_confidence(
        np.exp(-fb_errors[finite_fb] / max(float(config.forward_backward_scale_px), 1e-12)),
        config,
    )

    local_conf[valid_mask] = clip_confidence(speckle_conf[valid_mask] * motion_conf[valid_mask] * fb_conf[valid_mask], config)
    median_fb_error = float(np.nanmedian(fb_errors[finite_fb])) if np.any(finite_fb) else np.nan
    return local_conf, speckle_conf, motion_conf, fb_conf, d_med, robust_mad(residuals[valid_mask]), median_fb_error

In [5]:
def interval_score(value: float, lower: float, upper: float) -> float:
    if not np.isfinite(value):
        return 1.0
    if lower <= value <= upper:
        return 1.0
    span = max(float(upper - lower), 1e-6)
    distance = lower - value if value < lower else value - upper
    return float(np.exp(-distance / max(0.25 * span, 1e-6)))


def segment_length(segment: np.ndarray) -> float:
    arr = np.asarray(segment, dtype=float).reshape(-1)
    if arr.size < 4 or not np.all(np.isfinite(arr[:4])):
        return np.nan
    return float(np.hypot(arr[2] - arr[0], arr[3] - arr[1]))


def geometry_scores(frame_idx: int, config: SpeckleConfidenceConfig):
    alpha = float(timtrack_alpha[frame_idx]) if np.isfinite(timtrack_alpha[frame_idx]) else np.nan
    length = segment_length(fascicle_prior[frame_idx])
    alpha_score = interval_score(alpha, *config.plausible_alpha_range_deg)
    length_score = interval_score(length, *config.plausible_length_range_px)
    if frame_idx > 0:
        prev_alpha = float(timtrack_alpha[frame_idx - 1]) if np.isfinite(timtrack_alpha[frame_idx - 1]) else np.nan
        prev_length = segment_length(fascicle_prior[frame_idx - 1])
        angle_jump = normalize_angle_delta_deg(alpha, prev_alpha)
        length_jump = abs(length - prev_length) if np.isfinite(length) and np.isfinite(prev_length) else np.nan
        angle_jump_score = float(np.exp(-angle_jump / max(float(config.angle_jump_scale_deg), 1e-12))) if np.isfinite(angle_jump) else 1.0
        length_jump_score = float(np.exp(-length_jump / max(float(config.length_jump_scale_px), 1e-12))) if np.isfinite(length_jump) else 1.0
    else:
        angle_jump = np.nan
        length_jump = np.nan
        angle_jump_score = 1.0
        length_jump_score = 1.0
    scores = np.asarray([alpha_score, length_score, angle_jump_score, length_jump_score], dtype=float)
    scores = clip_confidence(scores[np.isfinite(scores)], config)
    geometry_stability = float(np.exp(np.mean(np.log(scores)))) if scores.size else 1.0
    return {
        'geometry_stability': float(clip_confidence(geometry_stability, config)),
        'geometry_alpha_score': float(clip_confidence(alpha_score, config)),
        'geometry_length_score': float(clip_confidence(length_score, config)),
        'geometry_angle_jump_deg': float(angle_jump),
        'geometry_angle_jump_score': float(clip_confidence(angle_jump_score, config)),
        'geometry_length_jump_px': float(length_jump),
        'geometry_length_jump_score': float(clip_confidence(length_jump_score, config)),
    }


def aponeurosis_stability(frame_idx: int, config: SpeckleConfidenceConfig):
    if frame_idx == 0:
        return 1.0
    sup_now, deep_now = super_lines[frame_idx], deep_lines[frame_idx]
    sup_prev, deep_prev = super_lines[frame_idx - 1], deep_lines[frame_idx - 1]
    sup_angle_jump = normalize_angle_delta_deg(line_angle_deg(sup_now), line_angle_deg(sup_prev))
    deep_angle_jump = normalize_angle_delta_deg(line_angle_deg(deep_now), line_angle_deg(deep_prev))
    y_jump = abs(np.mean(sup_now[[1, 3]]) - np.mean(sup_prev[[1, 3]])) + abs(np.mean(deep_now[[1, 3]]) - np.mean(deep_prev[[1, 3]]))
    angle_score = np.exp(-np.nanmean([sup_angle_jump, deep_angle_jump]) / max(float(config.angle_jump_scale_deg), 1e-12))
    y_score = np.exp(-y_jump / max(float(config.length_jump_scale_px), 1e-12))
    return float(clip_confidence(np.sqrt(angle_score * y_score), config))


def intersection_quality(frame_idx: int, config: SpeckleConfidenceConfig):
    segment = np.asarray(fascicle_prior[frame_idx], dtype=float).reshape(4)
    sup = super_lines[frame_idx]
    deep = deep_lines[frame_idx]
    pts = [line_intersection(segment, sup), line_intersection(segment, deep)]
    if any(p is None for p in pts):
        return float(config.confidence_floor)
    scores = []
    for x, y in pts:
        dx = max(0.0, -x, x - image_w)
        dy = max(0.0, -y, y - image_h)
        scores.append(np.exp(-np.hypot(dx, dy) / max(float(config.length_jump_scale_px), 1e-12)))
    seg_angle = line_angle_deg(segment)
    sep_sup = abs(np.sin(np.deg2rad(seg_angle - line_angle_deg(sup))))
    sep_deep = abs(np.sin(np.deg2rad(seg_angle - line_angle_deg(deep))))
    separation_score = np.clip(min(sep_sup, sep_deep) / 0.2, 0.0, 1.0)
    quality = float(np.prod(scores) ** 0.5 * separation_score)
    return float(clip_confidence(quality, config))

## 4. Compute or Load Full-Video Confidence

In [6]:
def iter_gray_frames(path: Path, limit: int):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(path)
    count = 0
    try:
        while count < limit:
            ok, frame = cap.read()
            if not ok:
                break
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
            yield gray
            count += 1
    finally:
        cap.release()


def empty_confidence_arrays(n: int):
    return {
        'combined_confidence': np.ones(n, dtype=np.float32),
        'confidence_theta': np.ones(n, dtype=np.float32),
        'confidence_length': np.ones(n, dtype=np.float32),
        'r_scale': np.ones(n, dtype=np.float32),
        'r_scale_theta': np.ones(n, dtype=np.float32),
        'r_scale_length': np.ones(n, dtype=np.float32),
        'speckle_confidence': np.ones(n, dtype=np.float32),
        'motion_consistency': np.ones(n, dtype=np.float32),
        'forward_backward_confidence': np.ones(n, dtype=np.float32),
        'median_zncc': np.full(n, np.nan, dtype=np.float32),
        'valid_fraction': np.zeros(n, dtype=np.float32),
        'median_forward_backward_error': np.full(n, np.nan, dtype=np.float32),
        'displacement_mad': np.full(n, np.nan, dtype=np.float32),
        'median_dx': np.full(n, np.nan, dtype=np.float32),
        'median_dy': np.full(n, np.nan, dtype=np.float32),
        'geometry_stability': np.ones(n, dtype=np.float32),
        'geometry_alpha_score': np.ones(n, dtype=np.float32),
        'geometry_length_score': np.ones(n, dtype=np.float32),
        'geometry_angle_jump_score': np.ones(n, dtype=np.float32),
        'geometry_length_jump_score': np.ones(n, dtype=np.float32),
        'aponeurosis_stability': np.ones(n, dtype=np.float32),
        'intersection_quality': np.ones(n, dtype=np.float32),
        'n_band_points': np.zeros(n, dtype=np.int32),
        'n_valid_points': np.zeros(n, dtype=np.int32),
    }


def compute_frame_confidence(frame_idx: int, prev_frame: np.ndarray | None, curr_frame: np.ndarray, base_points: np.ndarray, config: SpeckleConfidenceConfig):
    geom = geometry_scores(frame_idx, config)
    apo_conf = aponeurosis_stability(frame_idx, config)
    inter_conf = intersection_quality(frame_idx, config)
    if frame_idx == 0 or prev_frame is None:
        metrics = {
            'speckle_confidence': 1.0,
            'motion_consistency': 1.0,
            'feature_reliability': 1.0,
            'geometry_stability': geom['geometry_stability'],
            'geometry_alpha_score': geom['geometry_alpha_score'],
            'geometry_angle_jump_score': geom['geometry_angle_jump_score'],
            'geometry_length_score': geom['geometry_length_score'],
            'geometry_length_jump_score': geom['geometry_length_jump_score'],
            'aponeurosis_stability': apo_conf,
            'intersection_quality': inter_conf,
        }
        anis = combine_anisotropic_confidence_metrics(metrics, config=config)
        scales = anisotropic_confidence_to_r_scales(anis['confidence_theta'], anis['confidence_length'], config)
        return {
            **geom,
            'aponeurosis_stability': apo_conf,
            'intersection_quality': inter_conf,
            'combined_confidence': 1.0,
            'confidence_theta': anis['confidence_theta'],
            'confidence_length': anis['confidence_length'],
            'r_scale': confidence_to_r_scale(1.0, config),
            'r_scale_theta': scales['r_scale_theta'],
            'r_scale_length': scales['r_scale_length'],
            'speckle_confidence': 1.0,
            'motion_consistency': 1.0,
            'forward_backward_confidence': 1.0,
            'median_zncc': np.nan,
            'valid_fraction': 0.0,
            'median_forward_backward_error': np.nan,
            'displacement_mad': np.nan,
            'median_dx': np.nan,
            'median_dy': np.nan,
            'n_band_points': 0,
            'n_valid_points': 0,
            'points': np.empty((0, 2), dtype=np.float32),
            'displacements': np.empty((0, 2), dtype=np.float32),
            'valid': np.zeros(0, dtype=bool),
            'local_confidence': np.zeros(0, dtype=np.float32),
        }

    in_band = muscle_band_mask(base_points, super_lines[frame_idx - 1], deep_lines[frame_idx - 1], margin=3.0)
    points = base_points[in_band]
    disp, zncc, fb_error, valid = block_match_forward_backward(prev_frame, curr_frame, points, config)
    local_conf, speckle_conf, motion_conf, fb_conf, d_med, d_mad, fb_med = local_confidence_from_matches(
        disp, zncc, fb_error, valid, config
    )
    valid_fraction_frame = float(np.sum(valid) / max(len(points), 1))
    c_t = float(np.nanmedian(local_conf[valid])) if np.any(valid) else float(config.confidence_floor)
    if valid_fraction_frame < VALID_FRACTION_FLOOR:
        c_t *= valid_fraction_frame / VALID_FRACTION_FLOOR
    c_t = float(clip_confidence(c_t, config))

    speckle_frame = float(np.nanmedian(speckle_conf[valid] * fb_conf[valid])) if np.any(valid) else float(config.confidence_floor)
    motion_frame = float(np.nanmedian(motion_conf[valid])) if np.any(valid) else float(config.confidence_floor)
    fb_frame = float(np.nanmedian(fb_conf[valid])) if np.any(valid) else float(config.confidence_floor)
    metrics = {
        'speckle_confidence': speckle_frame,
        'motion_consistency': motion_frame,
        'feature_reliability': 1.0,
        'geometry_stability': geom['geometry_stability'],
        'geometry_alpha_score': geom['geometry_alpha_score'],
        'geometry_angle_jump_score': geom['geometry_angle_jump_score'],
        'geometry_length_score': geom['geometry_length_score'],
        'geometry_length_jump_score': geom['geometry_length_jump_score'],
        'aponeurosis_stability': apo_conf,
        'intersection_quality': inter_conf,
    }
    anis = combine_anisotropic_confidence_metrics(metrics, config=config)
    scales = anisotropic_confidence_to_r_scales(anis['confidence_theta'], anis['confidence_length'], config)
    return {
        **geom,
        'aponeurosis_stability': apo_conf,
        'intersection_quality': inter_conf,
        'combined_confidence': c_t,
        'confidence_theta': anis['confidence_theta'],
        'confidence_length': anis['confidence_length'],
        'r_scale': confidence_to_r_scale(c_t, config),
        'r_scale_theta': scales['r_scale_theta'],
        'r_scale_length': scales['r_scale_length'],
        'speckle_confidence': speckle_frame,
        'motion_consistency': motion_frame,
        'forward_backward_confidence': fb_frame,
        'median_zncc': float(np.nanmedian(zncc[valid])) if np.any(valid) else np.nan,
        'valid_fraction': valid_fraction_frame,
        'median_forward_backward_error': fb_med,
        'displacement_mad': d_mad,
        'median_dx': float(d_med[0]) if np.all(np.isfinite(d_med)) else np.nan,
        'median_dy': float(d_med[1]) if np.all(np.isfinite(d_med)) else np.nan,
        'n_band_points': int(len(points)),
        'n_valid_points': int(np.sum(valid)),
        'points': points,
        'displacements': disp,
        'valid': valid,
        'local_confidence': local_conf,
    }

In [7]:
def compute_confidence_series(n: int, config: SpeckleConfidenceConfig):
    cache_path = confidence_cache_path(n, config)
    if cache_path.exists() and not FORCE_RECOMPUTE_CONFIDENCE:
        data = dict(np.load(cache_path, allow_pickle=True))
        data['cache_path'] = str(cache_path)
        data['loaded_cache'] = True
        print('Loaded confidence cache:', cache_path)
        return data

    base_points = sample_roi_grid(fascicle_roi, config)
    arrays = empty_confidence_arrays(n)
    arrays['sample_points'] = base_points
    arrays['config_json'] = np.asarray(json.dumps(config_payload(config), sort_keys=True))

    prev_frame = None
    for idx, curr_frame in enumerate(iter_gray_frames(VIDEO, n)):
        frame_data = compute_frame_confidence(idx, prev_frame, curr_frame, base_points, config)
        for key in arrays:
            if key in {'sample_points', 'config_json'}:
                continue
            if key in frame_data:
                arrays[key][idx] = frame_data[key]
        prev_frame = curr_frame
        if idx > 0 and (idx % 250 == 0 or idx == n - 1):
            print(f'confidence {idx + 1}/{n}')

    np.savez_compressed(cache_path, **arrays)
    arrays['cache_path'] = str(cache_path)
    arrays['loaded_cache'] = False
    print('Saved confidence cache:', cache_path)
    return arrays


conf = compute_confidence_series(n_all, CONFIG)
print({
    'loaded_cache': bool(conf['loaded_cache']),
    'confidence_median': float(np.nanmedian(conf['combined_confidence'])),
    'theta_confidence_median': float(np.nanmedian(conf['confidence_theta'])),
    'length_confidence_median': float(np.nanmedian(conf['confidence_length'])),
    'r_scale_median': float(np.nanmedian(conf['r_scale'])),
    'r_scale_theta_median': float(np.nanmedian(conf['r_scale_theta'])),
    'r_scale_length_median': float(np.nanmedian(conf['r_scale_length'])),
})

confidence 251/2667


confidence 501/2667


confidence 751/2667


confidence 1001/2667


confidence 1251/2667


...

confidence 1751/2667


confidence 2001/2667


confidence 2251/2667


confidence 2501/2667


confidence 2667/2667
Saved confidence cache: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/confidence_cache/strict_conf_r4_c4e9f61590_first_2667_frames.npz
{'loaded_cache': False, 'confidence_median': 0.6065306663513184, 'theta_confidence_median': 0.9814246892929077, 'length_confidence_median': 0.9773587584495544, 'r_scale_median': 5.312834739685059, 'r_scale_theta_median': 0.5493672490119934, 'r_scale_length_median': 0.5664332509040833}


## 5. Run Scalar and Anisotropic Adaptive Kalman

In [8]:
mat_root = loadmat(UTT_EXPORT, simplify_cells=True)['UTT_numeric_export']
r_values = np.asarray(mat_root.get('R', [3.05529211]), dtype=float).reshape(-1)
kalman_cfg = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root.get('Q', 0.01)),
    x_measurement_variance=float(mat_root.get('X', 100.0)),
    alpha_measurement_variance=float(r_values[0]),
    n_start_frames=int(mat_root.get('NS', 1)),
    run_smoother=True,
    use_adaptive_R=True,
)
mm_per_px = float(np.asarray(baseline['mm_per_pixel']).reshape(-1)[0])

scalar = run_matlab_2state_kalman(
    fascicle_prior,
    timtrack_alpha,
    super_lines,
    deep_lines,
    config=kalman_cfg,
    mm_per_pixel=mm_per_px,
    measurement_r_scale=np.asarray(conf['r_scale'], dtype=float),
)
anisotropic = run_matlab_2state_kalman(
    fascicle_prior,
    timtrack_alpha,
    super_lines,
    deep_lines,
    config=kalman_cfg,
    mm_per_pixel=mm_per_px,
    measurement_r_scale=np.asarray(conf['r_scale'], dtype=float),
    measurement_r_scale_theta=np.asarray(conf['r_scale_theta'], dtype=float),
    measurement_r_scale_length=np.asarray(conf['r_scale_length'], dtype=float),
)

print('scalar measurement_R_diag frame 1:', scalar['measurement_R_diag'][1])
print('anisotropic measurement_R_diag frame 1:', anisotropic['measurement_R_diag'][1])

scalar measurement_R_diag frame 1: [50.          1.52764606]
anisotropic measurement_R_diag frame 1: [160.30396223   1.52764606]


## 6. Summaries: First 30 s and Full Video

In [9]:
def rmse(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.sqrt(np.nanmean((a - b) ** 2)))


def summarize_window(label: str, end: int):
    sl = slice(0, end)
    fixed_ang = np.asarray(baseline['ANG_deg'][sl], dtype=float)
    fixed_pen = np.asarray(baseline['PEN_deg'][sl], dtype=float)
    fixed_fl = np.asarray(baseline['FL_mm'][sl], dtype=float)
    scalar_ang = np.asarray(scalar['ANG_deg'][sl], dtype=float)
    scalar_pen = np.asarray(scalar['PEN_deg'][sl], dtype=float)
    scalar_fl = np.asarray(scalar['FL_mm'][sl], dtype=float)
    aniso_ang = np.asarray(anisotropic['ANG_deg'][sl], dtype=float)
    aniso_pen = np.asarray(anisotropic['PEN_deg'][sl], dtype=float)
    aniso_fl = np.asarray(anisotropic['FL_mm'][sl], dtype=float)
    r_scalar = np.asarray(conf['r_scale'][sl], dtype=float)
    r_theta = np.asarray(conf['r_scale_theta'][sl], dtype=float)
    r_length = np.asarray(conf['r_scale_length'][sl], dtype=float)
    R_theta = np.asarray(anisotropic['measurement_R_diag'][sl, 1], dtype=float)
    R_length = np.asarray(anisotropic['measurement_R_diag'][sl, 0], dtype=float)
    row = {
        'window': label,
        'frames': end,
        'seconds': float(time_all[end - 1]),
        'scalar_ANG_rmse_vs_fixed_deg': rmse(scalar_ang, fixed_ang),
        'anisotropic_ANG_rmse_vs_fixed_deg': rmse(aniso_ang, fixed_ang),
        'anisotropic_vs_scalar_ANG_rmse_deg': rmse(aniso_ang, scalar_ang),
        'scalar_PEN_rmse_vs_fixed_deg': rmse(scalar_pen, fixed_pen),
        'anisotropic_PEN_rmse_vs_fixed_deg': rmse(aniso_pen, fixed_pen),
        'scalar_FL_rmse_vs_fixed_mm': rmse(scalar_fl, fixed_fl),
        'anisotropic_FL_rmse_vs_fixed_mm': rmse(aniso_fl, fixed_fl),
        'anisotropic_vs_scalar_FL_rmse_mm': rmse(aniso_fl, scalar_fl),
        'confidence_median': float(np.nanmedian(conf['combined_confidence'][sl])),
        'confidence_theta_median': float(np.nanmedian(conf['confidence_theta'][sl])),
        'confidence_length_median': float(np.nanmedian(conf['confidence_length'][sl])),
        'r_scale_scalar_median': float(np.nanmedian(r_scalar)),
        'r_scale_scalar_q90': float(np.nanquantile(r_scalar, 0.90)),
        'r_scale_theta_median': float(np.nanmedian(r_theta)),
        'r_scale_theta_q90': float(np.nanquantile(r_theta, 0.90)),
        'r_scale_length_median': float(np.nanmedian(r_length)),
        'r_scale_length_q90': float(np.nanquantile(r_length, 0.90)),
        'R_theta_median': float(np.nanmedian(R_theta)),
        'R_theta_q90': float(np.nanquantile(R_theta, 0.90)),
        'R_L_median': float(np.nanmedian(R_length)),
        'R_L_q90': float(np.nanquantile(R_length, 0.90)),
    }
    return row

summary = pd.DataFrame([summarize_window('first30s', n_first), summarize_window('full_video', n_all)])
summary_path = OUT / 'scalar_vs_anisotropic_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/scalar_vs_anisotropic_summary.csv


,window,frames,seconds,scalar_ANG_rmse_vs_fixed_deg,anisotropic_ANG_rmse_vs_fixed_deg,anisotropic_vs_scalar_ANG_rmse_deg,scalar_PEN_rmse_vs_fixed_deg,anisotropic_PEN_rmse_vs_fixed_deg,scalar_FL_rmse_vs_fixed_mm,anisotropic_FL_rmse_vs_fixed_mm,...,r_scale_scalar_median,r_scale_scalar_q90,r_scale_theta_median,r_scale_theta_q90,r_scale_length_median,r_scale_length_q90,R_theta_median,R_theta_q90,R_L_median,R_L_q90
0,first30s,1000,29.963108,0.054505,0.195312,0.205015,0.054505,0.195312,0.108814,0.323425,...,5.312835,11.290354,0.555915,2.066345,0.706224,1.403395,1.698483,6.313288,70.622376,140.339493
1,full_video,2667,79.961609,0.128720,0.138112,0.157979,0.128720,0.138112,0.288730,0.264459,...,5.312835,11.373881,0.549367,2.079934,0.566433,0.846426,1.678477,6.354806,56.643325,84.642586


## 7. R-scale and Output Plots

In [10]:
def plot_window(label: str, end: int):
    sl = slice(0, end)
    t = time_all[sl]
    fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
    axes[0].plot(t, conf['combined_confidence'][sl], color='tab:purple', lw=1.0, label='scalar confidence')
    axes[0].plot(t, conf['confidence_theta'][sl], color='tab:red', lw=0.9, alpha=0.8, label='theta confidence')
    axes[0].plot(t, conf['confidence_length'][sl], color='tab:green', lw=0.9, alpha=0.8, label='length confidence')
    axes[0].set_ylabel('confidence')
    axes[0].set_ylim(0, 1.05)
    axes[0].legend(ncol=3, fontsize=8)

    axes[1].plot(t, conf['r_scale'][sl], color='tab:orange', lw=1.0, label='scalar r_scale')
    axes[1].plot(t, conf['r_scale_theta'][sl], color='tab:red', lw=0.9, alpha=0.8, label='r_scale_theta')
    axes[1].plot(t, conf['r_scale_length'][sl], color='tab:green', lw=0.9, alpha=0.8, label='r_scale_length')
    axes[1].set_ylabel('R scale')
    axes[1].legend(ncol=3, fontsize=8)

    axes[2].plot(t, baseline['ANG_deg'][sl], color='black', lw=1.0, label='fixed')
    axes[2].plot(t, scalar['ANG_deg'][sl], color='tab:orange', lw=0.9, label='scalar adaptive')
    axes[2].plot(t, anisotropic['ANG_deg'][sl], color='tab:red', lw=0.9, label='anisotropic adaptive')
    axes[2].set_ylabel('ANG deg')
    axes[2].legend(ncol=3, fontsize=8)

    axes[3].plot(t, baseline['FL_mm'][sl], color='black', lw=1.0, label='fixed')
    axes[3].plot(t, scalar['FL_mm'][sl], color='tab:orange', lw=0.9, label='scalar adaptive')
    axes[3].plot(t, anisotropic['FL_mm'][sl], color='tab:green', lw=0.9, label='anisotropic adaptive')
    axes[3].set_ylabel('FL mm')
    axes[3].set_xlabel('Time s')
    axes[3].legend(ncol=3, fontsize=8)
    for ax in axes:
        ax.grid(True, alpha=0.25)
    fig.suptitle(f'Scalar vs anisotropic adaptive R, {label}')
    fig.tight_layout()
    path = OUT / f'scalar_vs_anisotropic_{label}.png'
    fig.savefig(path, dpi=170)
    plt.close(fig)
    return path

paths = [plot_window('first30s', n_first), plot_window('full_video', n_all)]
paths

[PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/scalar_vs_anisotropic_first30s.png'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/scalar_vs_anisotropic_full_video.png')]

In [11]:
def plot_R_variances(label: str, end: int):
    sl = slice(0, end)
    t = time_all[sl]
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(t, scalar['measurement_R_diag'][sl, 1], color='tab:orange', lw=1.0, label='scalar R_theta')
    axes[0].plot(t, anisotropic['measurement_R_diag'][sl, 1], color='tab:red', lw=0.9, label='anisotropic R_theta')
    axes[0].set_ylabel('R_theta variance')
    axes[0].legend(fontsize=8)
    axes[1].plot(t, scalar['measurement_R_diag'][sl, 0], color='tab:orange', lw=1.0, label='scalar R_L')
    axes[1].plot(t, anisotropic['measurement_R_diag'][sl, 0], color='tab:green', lw=0.9, label='anisotropic R_L')
    axes[1].set_ylabel('R_L / x variance')
    axes[1].set_xlabel('Time s')
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.grid(True, alpha=0.25)
    fig.suptitle(f'Actual Kalman measurement variances, {label}')
    fig.tight_layout()
    path = OUT / f'Rtheta_RL_variances_{label}.png'
    fig.savefig(path, dpi=170)
    plt.close(fig)
    return path

variance_paths = [plot_R_variances('first30s', n_first), plot_R_variances('full_video', n_all)]
variance_paths

[PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/Rtheta_RL_variances_first30s.png'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/Rtheta_RL_variances_full_video.png')]

## 8. Low-confidence Frame Overlays

Overlay colors use local scalar block confidence. Text reports scalar/theta/length confidence and `R` scales.

In [12]:
def selected_low_confidence_frames(max_frame: int):
    candidates = []
    arrays = {
        'scalar_low': np.asarray(conf['combined_confidence'][:max_frame], dtype=float),
        'theta_low': np.asarray(conf['confidence_theta'][:max_frame], dtype=float),
        'length_low': np.asarray(conf['confidence_length'][:max_frame], dtype=float),
        'theta_minus_length_high': -(np.asarray(conf['r_scale_theta'][:max_frame], dtype=float) - np.asarray(conf['r_scale_length'][:max_frame], dtype=float)),
        'length_minus_theta_high': -(np.asarray(conf['r_scale_length'][:max_frame], dtype=float) - np.asarray(conf['r_scale_theta'][:max_frame], dtype=float)),
    }
    for reason, values in arrays.items():
        start = 1 if len(values) > 1 else 0
        idx = int(start + np.nanargmin(values[start:]))
        candidates.append((reason, idx))
    unique = []
    seen = set()
    for reason, idx in candidates:
        if idx not in seen:
            unique.append((reason, idx))
            seen.add(idx)
    return unique


def render_overlay(frame_idx: int, reason: str):
    prev_frame = read_gray_frame(VIDEO, max(frame_idx - 1, 0)) if frame_idx > 0 else None
    curr_frame = read_gray_frame(VIDEO, frame_idx)
    base_points = np.asarray(conf['sample_points'], dtype=np.float32)
    frame_data = compute_frame_confidence(frame_idx, prev_frame, curr_frame, base_points, CONFIG)

    vis = cv2.cvtColor(curr_frame, cv2.COLOR_GRAY2RGB)
    x, y, w, h = fascicle_roi
    cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 220, 0), 2)
    for point, d, c, ok in zip(frame_data['points'], frame_data['displacements'], frame_data['local_confidence'], frame_data['valid']):
        if not ok or not np.all(np.isfinite(d)):
            continue
        p0 = tuple(np.rint(point).astype(int))
        p1 = tuple(np.rint(point + 3.0 * d).astype(int))
        color = (int(255 * (1 - c)), int(255 * c), 40)
        cv2.circle(vis, p0, 2, color, -1)
        cv2.line(vis, p0, p1, color, 1)

    label = (
        f'{reason} frame {frame_idx} t={time_all[frame_idx]:.2f}s  '
        f'c={conf["combined_confidence"][frame_idx]:.2f} '
        f'cT={conf["confidence_theta"][frame_idx]:.2f} '
        f'cL={conf["confidence_length"][frame_idx]:.2f} '
        f'sT={conf["r_scale_theta"][frame_idx]:.1f} '
        f'sL={conf["r_scale_length"][frame_idx]:.1f}'
    )
    cv2.putText(vis, label, (24, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(vis, label, (24, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (0, 0, 0), 1, cv2.LINE_AA)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(vis)
    ax.set_axis_off()
    ax.set_title(label)
    fig.tight_layout()
    path = OVERLAY_DIR / f'{reason}_frame_{frame_idx:04d}.png'
    fig.savefig(path, dpi=170)
    plt.close(fig)
    return path

selected_frames = selected_low_confidence_frames(n_all)
overlay_paths = [render_overlay(idx, reason) for reason, idx in selected_frames]
overlay_table = pd.DataFrame([
    {
        'reason': reason,
        'frame': idx,
        'time_s': float(time_all[idx]),
        'confidence': float(conf['combined_confidence'][idx]),
        'confidence_theta': float(conf['confidence_theta'][idx]),
        'confidence_length': float(conf['confidence_length'][idx]),
        'r_scale_theta': float(conf['r_scale_theta'][idx]),
        'r_scale_length': float(conf['r_scale_length'][idx]),
        'path': str(path),
    }
    for (reason, idx), path in zip(selected_frames, overlay_paths)
])
overlay_csv = OVERLAY_DIR / 'selected_low_confidence_overlays.csv'
overlay_table.to_csv(overlay_csv, index=False)
print('Saved:', overlay_csv)
overlay_table

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/low_confidence_overlays/selected_low_confidence_overlays.csv


,reason,frame,time_s,confidence,confidence_theta,confidence_length,r_scale_theta,r_scale_length,path
0,scalar_low,189,5.668696,0.117548,0.657730,0.908185,4.404701,1.042509,/Users/grosbedou/PycharmProjects/NDORMS/result...
1,theta_low,1809,54.257521,0.119702,0.595800,0.960571,5.511059,0.652671,/Users/grosbedou/PycharmProjects/NDORMS/result...
2,length_low,40,1.199724,0.267905,0.736422,0.838728,3.138750,1.762916,/Users/grosbedou/PycharmProjects/NDORMS/result...
3,length_minus_theta_high,27,0.809814,1.000000,0.993769,0.847237,0.509590,1.664291,/Users/grosbedou/PycharmProjects/NDORMS/result...


## 9. Compact Interpretation

In [13]:
lines = []
for _, row in summary.iterrows():
    lines.append(
        f"{row['window']}: scalar ANG RMSE={row['scalar_ANG_rmse_vs_fixed_deg']:.4f} deg, "
        f"anisotropic ANG RMSE={row['anisotropic_ANG_rmse_vs_fixed_deg']:.4f} deg; "
        f"scalar FL RMSE={row['scalar_FL_rmse_vs_fixed_mm']:.4f} mm, "
        f"anisotropic FL RMSE={row['anisotropic_FL_rmse_vs_fixed_mm']:.4f} mm."
    )
lines.append('')
lines.append('R-scale medians/full video:')
full = summary[summary['window'] == 'full_video'].iloc[0]
lines.append(
    f"scalar={full['r_scale_scalar_median']:.2f}, theta={full['r_scale_theta_median']:.2f}, length={full['r_scale_length_median']:.2f}; "
    f"q90 scalar={full['r_scale_scalar_q90']:.2f}, theta={full['r_scale_theta_q90']:.2f}, length={full['r_scale_length_q90']:.2f}."
)
interpretation_path = OUT / 'scalar_vs_anisotropic_interpretation.txt'
interpretation_path.write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))
print('Saved:', interpretation_path)

first30s: scalar ANG RMSE=0.0545 deg, anisotropic ANG RMSE=0.1953 deg; scalar FL RMSE=0.1088 mm, anisotropic FL RMSE=0.3234 mm.
full_video: scalar ANG RMSE=0.1287 deg, anisotropic ANG RMSE=0.1381 deg; scalar FL RMSE=0.2887 mm, anisotropic FL RMSE=0.2645 mm.

R-scale medians/full video:
scalar=5.31, theta=0.55, length=0.57; q90 scalar=11.37, theta=2.08, length=0.85.
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook66_scalar_vs_anisotropic_adaptive_R_strict_conf_r4/scalar_vs_anisotropic_interpretation.txt
